# Dépendances

A exécuter seulement sur Colab, en local, les dépendances sont gérées avec uv.lock

In [8]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
%pip install -U transformers datasets peft trl accelerate safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 11.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
  Attempting uninstall: peft
    Found existing installation: peft 0.21.1
    Uninstalling peft-0.21.

In [1]:
import os
import subprocess
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO_NAME = "NLP_narrative-game"
REPO_URL = "https://github.com/2ec3/NLP_narrative-game.git"

if IN_COLAB:
    PROJECT_ROOT = Path("/content") / REPO_NAME
    if not PROJECT_ROOT.exists():
        if "TON_UTILISATEUR" in REPO_URL or "TON_DEPOT" in REPO_URL:
            raise ValueError("Modifie REPO_URL avec l'URL GitHub réelle avant de cloner le dépôt dans Colab.")
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    PROJECT_ROOT = next((p for p in candidates if (p / "datasets").is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Impossible de trouver un dossier datasets/ depuis {current}. "
            "Lance le notebook depuis le projet ou modifie PROJECT_ROOT manuellement."
        )

if not (PROJECT_ROOT / "datasets").is_dir():
    raise FileNotFoundError(
        f"Le dossier {PROJECT_ROOT / 'datasets'} n'existe pas. "
        "Vérifie que les données sont présentes dans le dépôt ou accessibles depuis le runtime."
    )

os.chdir(PROJECT_ROOT)
print("Environnement :", "Google Colab" if IN_COLAB else "local")
print("Racine du projet :", PROJECT_ROOT)
print("Données :", PROJECT_ROOT / "datasets")

SAVE_TO_DRIVE = IN_COLAB
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/GameMasterTraining")
else:
    OUTPUT_ROOT = PROJECT_ROOT / "model_adapter"

CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
ADAPTER_DIR = OUTPUT_ROOT / "best_gm_adapter"
MERGED_DIR = OUTPUT_ROOT / "merged_model"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Répertoire de sortie :", OUTPUT_ROOT)

Environnement : Google Colab
Racine du projet : /content/NLP_narrative-game
Données : /content/NLP_narrative-game/datasets
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Répertoire de sortie : /content/drive/MyDrive/GameMasterTraining


# Chargement du modèle et du tokenizer

In [7]:
import gc
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, EarlyStoppingCallback
from peft import LoraConfig, PeftModel
from datasets import Dataset, DatasetDict
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

USE_CUDA = torch.cuda.is_available()
USE_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if USE_CUDA:
    USE_BF16 = False
    USE_FP16 = True
    MODEL_DTYPE = torch.float16
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif USE_MPS:
    USE_BF16 = False
    USE_FP16 = False
    MODEL_DTYPE = torch.float32
    DEVICE_NAME = "Apple Metal (MPS)"
else:
    USE_BF16 = False
    USE_FP16 = False
    MODEL_DTYPE = torch.float32
    DEVICE_NAME = "CPU"

print("Matériel détecté :", DEVICE_NAME)
print("Précision du modèle :", MODEL_DTYPE)
print("CUDA disponible :", USE_CUDA)

# Reproductibilité de base
SEED = 42
torch.manual_seed(SEED)
if USE_CUDA:
    torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("Tokenizer et modèle chargés.")


Matériel détecté : Tesla T4
Précision du modèle : torch.float16
CUDA disponible : True


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizer et modèle chargés.


# Chargement des données

In [8]:
import json
from pathlib import Path
from datasets import Dataset, DatasetDict

TRAIN_FILE = PROJECT_ROOT / "datasets" / "train.json"
VALIDATION_FILE = PROJECT_ROOT / "datasets" / "validation.json"


def read_examples(file_path: Path):
    if not file_path.exists():
        raise FileNotFoundError(
            f"Fichier introuvable : {file_path}"
        )

    raw_text = file_path.read_text(encoding="utf-8").strip()

    if not raw_text:
        raise ValueError(f"Le fichier est vide : {file_path}")

    # Lecture JSON classique ou JSONL
    try:
        obj = json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            obj = [
                json.loads(line)
                for line in raw_text.splitlines()
                if line.strip()
            ]
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Format JSON/JSONL invalide dans {file_path}"
            ) from e

    # Récupérer les exemples
    if isinstance(obj, dict):
        if "examples" in obj:
            examples = obj["examples"]
        elif "messages" in obj:
            examples = [obj]
        else:
            raise ValueError(
                f"{file_path} doit contenir 'examples' ou 'messages'."
            )

    elif isinstance(obj, list):
        examples = obj

    else:
        raise ValueError(
            f"Format JSON non reconnu dans {file_path}."
        )

    if not isinstance(examples, list) or not examples:
        raise ValueError(f"Aucun exemple trouvé dans {file_path}.")

    cleaned_examples = []
    skipped_metadata = 0

    for i, example in enumerate(examples):

        if (
            not isinstance(example, dict)
            or not isinstance(example.get("messages"), list)
        ):
            raise ValueError(
                f"Exemple {i} : clé 'messages' manquante ou invalide."
            )

        cleaned_messages = []

        for j, message in enumerate(example["messages"]):

            if not isinstance(message, dict):
                raise ValueError(
                    f"Exemple {i}, message {j} : "
                    f"le message doit être un objet JSON."
                )

            # Ignorer les éléments de métadonnées placés
            # par erreur dans la liste des messages.
            if "type" in message and (
                "role" not in message or "content" not in message
            ):
                print(
                    f"Métadonnée ignorée : exemple {i}, "
                    f"message {j} : {message}"
                )
                skipped_metadata += 1
                continue

            # Vérifier les messages réellement destinés au modèle.
            if "role" not in message or "content" not in message:
                raise ValueError(
                    f"Exemple {i}, message {j} invalide : {message}"
                )

            # Ne garder que les champs utiles à l'entraînement.
            cleaned_message = {
                "role": message["role"],
                "content": message["content"]
            }

            if cleaned_message["role"] not in {
                "system", "user", "assistant"
            }:
                raise ValueError(
                    f"Rôle non pris en charge dans l'exemple {i} : "
                    f"{cleaned_message['role']}"
                )

            if not isinstance(cleaned_message["content"], str):
                raise ValueError(
                    f"Contenu non textuel dans l'exemple {i}, "
                    f"message {j}."
                )

            cleaned_messages.append(cleaned_message)

        if not cleaned_messages:
            raise ValueError(
                f"L'exemple {i} ne contient aucun message valide."
            )

        if cleaned_messages[-1]["role"] != "assistant":
            print(
                f"Attention : le dernier message de l'exemple {i} "
                f"n'est pas un message assistant."
            )

        # Conserver uniquement messages :
        # le champ type de l'exemple est ainsi supprimé.
        cleaned_examples.append({
            "messages": cleaned_messages
        })

    print(f"Fichier : {file_path.name}")
    print(f"Exemples conservés : {len(cleaned_examples)}")
    print(f"Métadonnées ignorées : {skipped_metadata}")

    return cleaned_examples


# Chargement et nettoyage
train_examples = read_examples(TRAIN_FILE)
validation_examples = read_examples(VALIDATION_FILE)

# Création des datasets Hugging Face
raw_dataset = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "validation": Dataset.from_list(validation_examples),
})

print(raw_dataset)
print("\nExemple brut :")
print(raw_dataset["train"][0]["messages"])

Métadonnée ignorée : exemple 182, message 4 : {'type': 'action.valide'}
Métadonnée ignorée : exemple 182, message 7 : {'type': 'action.valide'}
Métadonnée ignorée : exemple 182, message 10 : {'type': 'action.invalide'}
Fichier : train.json
Exemples conservés : 273
Métadonnées ignorées : 3
Fichier : validation.json
Exemples conservés : 40
Métadonnées ignorées : 0
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 273
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 40
    })
})

Exemple brut :
[{'role': 'system', 'content': "Tu es le Game Master d'un jeu de rôle fantasy en solo. Comprends l'intention du joueur lorsqu'elle est claire. Lorsqu'une formulation peut raisonnablement correspondre à plusieurs actions proposées, demande une précision plutôt que de choisir arbitrairement."}, {'role': 'assistant', 'content': 'Au fond de la salle se trouve un coffre en métal. Une serrure complexe protège son contenu. Vous pouvez : 

Appliquer le chat template de Qwen sur les datasets

In [9]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_splits = {}
for split_name, split_dataset in raw_dataset.items():
    formatted_splits[split_name] = split_dataset.map(
        format_example,
        remove_columns=split_dataset.column_names,
    )

dataset = DatasetDict(formatted_splits)

print("Exemple formaté :\n")
print(dataset["train"][0]["text"])
print("\nNombre de lignes train :", len(dataset["train"]))
print("Nombre de lignes validation :", len(dataset["validation"]))

Map:   0%|          | 0/273 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Exemple formaté :

<|im_start|>system
Tu es le Game Master d'un jeu de rôle fantasy en solo. Comprends l'intention du joueur lorsqu'elle est claire. Lorsqu'une formulation peut raisonnablement correspondre à plusieurs actions proposées, demande une précision plutôt que de choisir arbitrairement.<|im_end|>
<|im_start|>assistant
Au fond de la salle se trouve un coffre en métal. Une serrure complexe protège son contenu. Vous pouvez : A) examiner la serrure B) ouvrir le coffre C) quitter la salle.<|im_end|>
<|im_start|>user
Je m'occupe du coffre.<|im_end|>
<|im_start|>assistant
Vous vous approchez du coffre. Que souhaitez-vous faire ? A) examiner sa serrure B) tenter de l'ouvrir C) quitter la salle.<|im_end|>


Nombre de lignes train : 273
Nombre de lignes validation : 40


# Configuration de LoRA et de l'entraînement

In [10]:
# Configuration LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)

# Paramètres d'entraînement modifiables
QUICK_TEST = True  # mettre True pour vérifier le pipeline avec un entraînement très court
NUM_TRAIN_EPOCHS = 1 if QUICK_TEST else 3
MAX_LENGTH = 128 if QUICK_TEST else 512
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
USE_WANDB = False

if USE_WANDB:
    import wandb
    wandb.init(project="Mini-Projet-GameMaster", name="Qwen25-1.5B-LoRA")

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    run_name="Qwen25-1.5B-LoRA",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    max_length=MAX_LENGTH,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=True,
    report_to="wandb" if USE_WANDB else "none",
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

# Affiche le nombre de paramètres entraînables (ceux de l'adaptateur, principalement)
trainer.model.print_trainable_parameters()

Adding EOS to train dataset:   0%|          | 0/273 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/273 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/273 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/273 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/273 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


# Entraînement puis sauvegarde

In [11]:
try:
    print("Début de l'entraînement...")
    train_result = trainer.train()
    print(train_result)

    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print("Adaptateur LoRA sauvegardé dans :", ADAPTER_DIR)
finally:
    if USE_WANDB:
        wandb.finish()

Début de l'entraînement...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.779063,0.956981,0.993585,0.781693,34944.000000


TrainOutput(global_step=69, training_loss=1.1484434708305027, metrics={'train_runtime': 169.7775, 'train_samples_per_second': 1.608, 'train_steps_per_second': 0.406, 'total_flos': 276666951794688.0, 'train_loss': 1.1484434708305027, 'epoch': 1.0})
Adaptateur LoRA sauvegardé dans : /content/drive/MyDrive/GameMasterTraining/best_gm_adapter


## Test

In [12]:
test_messages = [
    {
        "role": "system",
        "content": (
            "Tu es le Game Master d'une aventure fantasy en solo. "
            "Décris les conséquences des actions en respectant l'état du jeu. "
            "Propose trois actions cohérentes lorsque c'est approprié."
        ),
    },
    {
        "role": "assistant",
        "content": (
            "Vous arrivez devant une porte en bois fermée. Une faible lumière "
            "passe sous la porte. Vous pouvez : A. Examiner la serrure ; "
            "B. Écouter derrière la porte ; C. Explorer le couloir."
        ),
    },
    {
        "role": "user",
        "content": "J'écoute derrière la porte.",
    },
]

trained_model = trainer.model
trained_model.eval()
input_batch = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
model_device = next(trained_model.parameters()).device
input_batch = input_batch.to(model_device)

with torch.inference_mode():
    output_ids = trained_model.generate(
        **input_batch,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

prompt_length = input_batch["input_ids"].shape[1]
answer = tokenizer.decode(output_ids[0][prompt_length:], skip_special_tokens=True)
print("--- Réponse du Game Master ---")
print(answer)

--- Réponse du Game Master ---
Un murmure vous parvient, comme si quelqu'un parlait dans un coin de la pièce. Vous pouvez : A. Laisser tomber votre oreille sur la porte pour entendre mieux ; B. Faire demi-tour et retourner à la sortie ; C. Continuer vers le couloir.
